# CrackFusion-Tiny — complete Colab pipeline

Run all cells in order. This single notebook performs Stages 1–5, downloads SDNET2018 from the supplied Google Drive link, trains and evaluates MobileViT-XS, creates the figures and reports, then downloads one ZIP containing every file under `/content/outputs`. A GPU runtime is strongly recommended.


In [ ]:
%cd /content
%pip install -q timm scikit-learn matplotlib gdown


# Stage 1 — MobileViT-XS setup and smoke test

This notebook implements Track A1-A2 from the project plan. It installs the minimum model dependency, selects Apple MPS/CUDA/CPU automatically, loads the ImageNet-pretrained MobileViT-XS checkpoint from the Hugging Face Hub through timm, uses the checkpoint's own preprocessing, checks a binary forward pass, and records intermediate feature shapes.

No crack dataset is used yet. Passing this notebook proves only that the model pipeline works; it is not an experimental result.

## 1. Install the minimum dependency

Run this once per environment. timm installs its required PyTorch and torchvision dependencies when they are absent.

## 2. Imports, seed, and device

The same notebook works on the local M4 Mac, a CUDA machine, or CPU.

In [ ]:
import json
from pathlib import Path

import timm
import torch
from PIL import Image
from timm.data import create_transform, resolve_model_data_config

SEED = 42
MODEL_NAME = "mobilevit_xs.cvnets_in1k"
OUTPUT_DIR = Path("outputs/mobilevit_xs")

torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print({
    "python_device": str(DEVICE),
    "torch": torch.__version__,
    "timm": timm.__version__,
})

## 3. Load MobileViT-XS for binary classification

The pretrained ImageNet classifier is replaced with one output logit. The rest of the checkpoint remains pretrained.

In [ ]:
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=1,
)
model = model.to(DEVICE).eval()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
classifier_parameters = sum(parameter.numel() for parameter in model.get_classifier().parameters())
backbone_parameters = total_parameters - classifier_parameters

print(model.__class__.__name__)
print(f"Binary model parameters: {total_parameters:,}")
print(f"Backbone parameters: {backbone_parameters:,}")
print(f"One-logit head parameters: {classifier_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print("Note: the published 2.3M count includes the original 1,000-class ImageNet head.")

assert 1_000_000 < total_parameters < 3_000_000, total_parameters
assert model.num_classes == 1
assert getattr(model.get_classifier(), "out_features", None) == 1

## 4. Resolve the checkpoint's preprocessing

Do not manually guess resize, crop, normalization, interpolation, or channel handling. timm supplies the configuration attached to the checkpoint.

In [ ]:
data_config = resolve_model_data_config(model)
inference_transform = create_transform(**data_config, is_training=False)

print(json.dumps({
    key: list(value) if isinstance(value, tuple) else value
    for key, value in data_config.items()
}, indent=2))
print(inference_transform)

## 5. Deterministic binary forward pass

Two synthetic PIL images are passed through the model-specific transform. This exercises the same input route that real crack images will use in Stage 2.

In [ ]:
images = [
    Image.new("RGB", (320, 280), color=(96, 96, 96)),
    Image.new("RGB", (280, 320), color=(160, 160, 160)),
]
batch = torch.stack([inference_transform(image) for image in images]).to(DEVICE)

with torch.inference_mode():
    logits = model(batch)

print("Input shape:", tuple(batch.shape))
print("Output shape:", tuple(logits.shape))
print("Logits:", logits.detach().cpu().flatten().tolist())

assert tuple(batch.shape) == (2, 3, 256, 256), tuple(batch.shape)
assert tuple(logits.shape) == (2, 1), tuple(logits.shape)
assert torch.isfinite(logits).all().item()

## 6. Record intermediate feature maps

These shapes are the first presentation artifact and show how spatial resolution decreases while channel depth grows through MobileViT.

In [ ]:
feature_model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    features_only=True,
).to(DEVICE).eval()

with torch.inference_mode():
    feature_maps = feature_model(batch)

feature_shapes = [list(feature.shape) for feature in feature_maps]
for index, shape in enumerate(feature_shapes, start=1):
    print(f"Stage {index}: {shape}")

assert feature_shapes
assert all(shape[0] == 2 for shape in feature_shapes)

## 7. Save the Stage 1 summary

The JSON output can be reused in the presentation and checked into the research artifacts after the notebook passes.

In [ ]:
summary = {
    "stage": "MobileViT-XS smoke test",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "device": str(DEVICE),
    "input_shape": list(batch.shape),
    "output_shape": list(logits.shape),
    "total_parameters": total_parameters,
    "backbone_parameters": backbone_parameters,
    "classifier_parameters": classifier_parameters,
    "trainable_parameters": trainable_parameters,
    "feature_shapes": feature_shapes,
    "data_config": {
        key: list(value) if isinstance(value, tuple) else value
        for key, value in data_config.items()
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / "stage_one_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

print(f"Saved {summary_path.resolve()}")

## Stage 1 exit criterion

Stage 1 is complete when every assertion passes and outputs/mobilevit_xs/stage_one_summary.json exists. The next notebook stage is SDNET2018 download, audit, class counts, and manifest creation.

# Stage 2 — SDNET2018 audit and manifest

This notebook implements Track A3-A4 from the MobileViT plan. It downloads the official SDNET2018 archive, verifies it before extraction, audits every image, records exact duplicates without deleting source files, and writes portable CSV manifests.

Parent-photo IDs are never guessed. If no verified parent mapping is supplied, the research split remains unset and the optional presentation split is labelled tile_development_only.

## 1. Dataset source and expected structure

Official record: https://digitalcommons.usu.edu/all_datasets/48/

Expected folders: D/CD, D/UD, P/CP, P/UP, W/CW, and W/UW. Labels use crack=1 and non-crack=0 so the positive class matches the later weighted BCE loss.

In [ ]:
from collections import Counter, defaultdict
import csv
import hashlib
import json
from pathlib import Path
import random
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from zipfile import ZipFile

from PIL import Image

SEED = 42
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
ARCHIVE_PATH = DATA_DIR / "SDNET2018.zip"
EXTRACT_DIR = DATA_DIR / "SDNET2018_extracted"
OUTPUT_DIR = WORK_DIR / "outputs" / "sdnet2018_audit"

OFFICIAL_RECORD_URL = "https://digitalcommons.usu.edu/all_datasets/48/"
ARCHIVE_URL = (
    "https://digitalcommons.usu.edu/cgi/viewcontent.cgi?"
    "article=1047&context=all_datasets&filename=2&type=additional"
)
EXPECTED_ARCHIVE_MD5 = "677411e784f194422c90f52d9ed0d7c6"

CLASS_INFO = {
    "CD": {"label": 1, "class_name": "crack", "surface": "deck"},
    "UD": {"label": 0, "class_name": "non_crack", "surface": "deck"},
    "CP": {"label": 1, "class_name": "crack", "surface": "pavement"},
    "UP": {"label": 0, "class_name": "non_crack", "surface": "pavement"},
    "CW": {"label": 1, "class_name": "crack", "surface": "wall"},
    "UW": {"label": 0, "class_name": "non_crack", "surface": "wall"},
}
EXPECTED_COUNTS = {
    "CD": 2025,
    "UD": 11595,
    "CP": 2608,
    "UP": 21726,
    "CW": 3851,
    "UW": 14287,
}
IMAGE_SUFFIXES = {".jpg", ".jpeg"}

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "work_dir": str(WORK_DIR.resolve()),
    "archive": str(ARCHIVE_PATH.resolve()),
    "official_record": OFFICIAL_RECORD_URL,
})

## 2. Download the official archive

The file is roughly 516 MB. Existing downloads are reused. If the host blocks automated access, download SDNET2018.zip from the official record and place it at the archive path printed above.

### VS Code + remote Colab fallback

The university host may return HTTP 403 to scripts. In that case:

1. Download SDNET2018.zip through your normal browser from the official record.
2. Upload the unchanged ZIP to Google Drive.
3. Set that Drive file to Anyone with the link — Viewer. The dataset is already public.
4. Paste its share URL below and run the cell. It copies the ZIP into the remote Colab runtime where the remaining notebook cells expect it.

Leave the value blank when the archive already exists.

In [ ]:
GOOGLE_DRIVE_SHARE_URL = "https://drive.google.com/file/d/1j3nTlWmrlL_5MJYwF95ztFcNz3F_hjfQ/view?usp=drive_link"

if GOOGLE_DRIVE_SHARE_URL and not ARCHIVE_PATH.exists():
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gdown"],
        check=True,
    )
    import gdown

    downloaded_path = gdown.download(
        url=GOOGLE_DRIVE_SHARE_URL,
        output=str(ARCHIVE_PATH),
        quiet=False,
        fuzzy=True,
    )
    if downloaded_path is None or not ARCHIVE_PATH.exists():
        raise RuntimeError("Google Drive download failed. Check that link access is Viewer for anyone with the link.")
elif ARCHIVE_PATH.exists():
    print(f"Archive already present: {ARCHIVE_PATH.resolve()}")
else:
    print("No Drive URL supplied; the next cell will try the official host.")

In [ ]:
def download_file(url, destination, chunk_size=1024 * 1024):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urlopen(request, timeout=60) as response, destination.open("wb") as output:
            total = int(response.headers.get("Content-Length", 0))
            downloaded = 0
            while chunk := response.read(chunk_size):
                output.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(
                        f"\rDownloaded {downloaded / 1024**2:.1f} / "
                        f"{total / 1024**2:.1f} MiB",
                        end="",
                    )
        print()
    except (HTTPError, URLError) as error:
        destination.unlink(missing_ok=True)
        raise RuntimeError(
            f"Automatic download failed: {error}. Download the archive from "
            f"{OFFICIAL_RECORD_URL} and place it at {destination.resolve()}."
        ) from error

if ARCHIVE_PATH.exists():
    print(f"Reusing {ARCHIVE_PATH.resolve()}")
else:
    download_file(ARCHIVE_URL, ARCHIVE_PATH)

print(f"Archive size: {ARCHIVE_PATH.stat().st_size / 1024**2:.1f} MiB")

## 3. Verify archive integrity

The official record publishes a 32-character checksum, treated here as MD5. A SHA-256 digest is also recorded for this project. Extraction stops if the published checksum does not match.

In [ ]:
def file_digest(path, algorithm, chunk_size=1024 * 1024):
    digest = hashlib.new(algorithm)
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

archive_md5 = file_digest(ARCHIVE_PATH, "md5")
archive_sha256 = file_digest(ARCHIVE_PATH, "sha256")

print("MD5:   ", archive_md5)
print("SHA256:", archive_sha256)
assert archive_md5 == EXPECTED_ARCHIVE_MD5, (
    "Archive checksum mismatch. Delete the file and download it again from the official record."
)

## 4. Safely extract and locate the dataset root

Archive members are checked against path traversal before extraction.

In [ ]:
def safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)

def find_dataset_root(extract_dir):
    required = [Path("D/CD"), Path("D/UD"), Path("P/CP"), Path("P/UP"), Path("W/CW"), Path("W/UW")]
    direct_candidates = [extract_dir, *[path for path in extract_dir.iterdir() if path.is_dir()]]
    for candidate in direct_candidates:
        if all((candidate / relative).is_dir() for relative in required):
            return candidate
    for cd_dir in extract_dir.rglob("CD"):
        if cd_dir.is_dir():
            candidate = cd_dir.parent.parent
            if all((candidate / relative).is_dir() for relative in required):
                return candidate
    raise FileNotFoundError("Could not find all six SDNET2018 class folders after extraction.")

try:
    DATASET_ROOT = find_dataset_root(EXTRACT_DIR)
    print("Reusing extracted dataset")
except (FileNotFoundError, StopIteration):
    safe_extract(ARCHIVE_PATH, EXTRACT_DIR)
    inner_archive = next(EXTRACT_DIR.rglob("SDNET2018.zip"), None)
    if inner_archive is not None:
        safe_extract(inner_archive, EXTRACT_DIR / "SDNET2018")
    DATASET_ROOT = find_dataset_root(EXTRACT_DIR)

print("Dataset root:", DATASET_ROOT.resolve())

## 5. Inventory class folders

Counts are compared with the original publication. A mismatch is reported rather than hidden.

In [ ]:
image_paths_by_code = {}
observed_counts = {}

for code in CLASS_INFO:
    surface_folder = {"D": "D", "P": "P", "W": "W"}[code[-1]]
    class_dir = DATASET_ROOT / surface_folder / code
    paths = sorted(
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    image_paths_by_code[code] = paths
    observed_counts[code] = len(paths)

print(f"{'Code':<5}{'Observed':>12}{'Expected':>12}{'Status':>12}")
for code in CLASS_INFO:
    status = "OK" if observed_counts[code] == EXPECTED_COUNTS[code] else "MISMATCH"
    print(f"{code:<5}{observed_counts[code]:>12,}{EXPECTED_COUNTS[code]:>12,}{status:>12}")

print(f"Total observed: {sum(observed_counts.values()):,}")
print(f"Total expected: {sum(EXPECTED_COUNTS.values()):,}")
assert sum(observed_counts.values()) >= 56_000

## 6. Audit every image and hash exact content

Files are checked for readability, 256×256 dimensions, RGB-compatible mode, and exact SHA-256 duplicates. Duplicate files remain on disk but all except the first canonical path are marked excluded in the manifest.

In [ ]:
rows = []
paths_by_digest = defaultdict(list)
all_paths = [path for code in CLASS_INFO for path in image_paths_by_code[code]]

for index, path in enumerate(all_paths, start=1):
    code = path.parent.name.upper()
    info = CLASS_INFO[code]
    relative_path = path.relative_to(DATASET_ROOT).as_posix()
    readable = True
    width = height = 0
    mode = ""
    error = ""

    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            width, height = image.size
            mode = image.mode
    except Exception as exception:
        readable = False
        error = f"{type(exception).__name__}: {exception}"

    excluded = not readable
    if readable and (width, height) != (256, 256):
        excluded = True
        error = f"unexpected_dimensions:{width}x{height}"
    elif readable and mode != "RGB":
        excluded = True
        error = f"unexpected_mode:{mode}"

    sha256 = file_digest(path, "sha256")
    paths_by_digest[sha256].append(relative_path)
    rows.append({
        "path": relative_path,
        "label": info["label"],
        "class_name": info["class_name"],
        "surface": info["surface"],
        "class_code": code,
        "parent_id": "",
        "sha256": sha256,
        "width": width,
        "height": height,
        "mode": mode,
        "readable": int(readable),
        "excluded": int(excluded),
        "exclusion_reason": error,
        "split": "",
        "split_basis": "unassigned",
    })

    if index % 5_000 == 0 or index == len(all_paths):
        print(f"Audited {index:,}/{len(all_paths):,}")

rows_by_path = {row["path"]: row for row in rows}
duplicate_groups = {
    digest: sorted(paths)
    for digest, paths in paths_by_digest.items()
    if len(paths) > 1
}

for duplicate_paths in duplicate_groups.values():
    canonical = duplicate_paths[0]
    for duplicate in duplicate_paths[1:]:
        row = rows_by_path[duplicate]
        row["excluded"] = 1
        row["exclusion_reason"] = f"exact_duplicate_of:{canonical}"

corrupt_count = sum(not row["readable"] for row in rows)
wrong_size_count = sum(
    row["readable"] and (row["width"], row["height"]) != (256, 256)
    for row in rows
)
non_rgb_count = sum(row["readable"] and row["mode"] != "RGB" for row in rows)
duplicate_file_count = sum(len(paths) - 1 for paths in duplicate_groups.values())

print({
    "corrupt": corrupt_count,
    "wrong_size": wrong_size_count,
    "non_rgb": non_rgb_count,
    "duplicate_groups": len(duplicate_groups),
    "duplicate_files_excluded": duplicate_file_count,
})

## 7. Add verified parent IDs if available

The public class folders do not by themselves prove which tiles came from the same original photograph. To add parent IDs, supply data/parent_mapping.csv with columns path,parent_id, where path matches the manifest-relative path such as D/CD/example.jpg. Otherwise parent_id remains blank.

In [ ]:
PARENT_MAPPING_PATH = DATA_DIR / "parent_mapping.csv"

if PARENT_MAPPING_PATH.exists():
    with PARENT_MAPPING_PATH.open(newline="", encoding="utf-8") as stream:
        parent_mapping = {
            item["path"]: item["parent_id"]
            for item in csv.DictReader(stream)
        }
    unknown_paths = sorted(set(parent_mapping) - set(rows_by_path))
    if unknown_paths:
        raise ValueError(f"Parent mapping contains unknown paths, starting with: {unknown_paths[:3]}")
    for row in rows:
        row["parent_id"] = parent_mapping.get(row["path"], "")
else:
    parent_mapping = {}

eligible_rows = [row for row in rows if not row["excluded"]]
parent_coverage = sum(bool(row["parent_id"]) for row in eligible_rows)
print(f"Verified parent IDs: {parent_coverage:,}/{len(eligible_rows):,}")

if parent_coverage != len(eligible_rows):
    print(
        "Research split is intentionally unassigned: complete verified parent mapping is unavailable."
    )

## 8. Write the audit manifest

This is the complete audit record. It includes excluded files and leaves the research split blank until parent grouping is defensible.

In [ ]:
MANIFEST_FIELDS = [
    "path", "label", "class_name", "surface", "class_code", "parent_id",
    "sha256", "width", "height", "mode", "readable", "excluded",
    "exclusion_reason", "split", "split_basis",
]
AUDIT_MANIFEST_PATH = OUTPUT_DIR / "audit_manifest.csv"

with AUDIT_MANIFEST_PATH.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda row: row["path"]))

print("Saved", AUDIT_MANIFEST_PATH.resolve())

## 9. Create the presentation-only development manifest

The default takes 600 usable images from each class/surface folder, giving 3,600 images balanced across label and surface. Each group is split 80/20. This is a tile-level pipeline check, not a research test.

In [ ]:
DEV_PER_CLASS_SURFACE = 600
DEV_TRAIN_FRACTION = 0.8
development_rows = []

eligible_by_code = defaultdict(list)
for row in eligible_rows:
    eligible_by_code[row["class_code"]].append(row)

for code in CLASS_INFO:
    candidates = sorted(eligible_by_code[code], key=lambda row: row["path"])
    if len(candidates) < DEV_PER_CLASS_SURFACE:
        raise ValueError(f"Not enough usable images in {code}: {len(candidates)}")
    rng = random.Random(SEED + sum(map(ord, code)))
    selected = rng.sample(candidates, DEV_PER_CLASS_SURFACE)
    split_at = round(len(selected) * DEV_TRAIN_FRACTION)
    for index, source_row in enumerate(selected):
        row = dict(source_row)
        row["split"] = "dev_train" if index < split_at else "dev_validation"
        row["split_basis"] = "tile_development_only"
        development_rows.append(row)

DEVELOPMENT_MANIFEST_PATH = OUTPUT_DIR / "development_manifest.csv"
with DEVELOPMENT_MANIFEST_PATH.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)
    writer.writeheader()
    writer.writerows(sorted(development_rows, key=lambda row: (row["split"], row["path"])))

dev_counts = Counter(
    (row["split"], row["label"], row["surface"])
    for row in development_rows
)
for key, count in sorted(dev_counts.items()):
    print(key, count)

print("Saved", DEVELOPMENT_MANIFEST_PATH.resolve())
print("WARNING: tile_development_only; possible parent-photo leakage; not a Phase 3 result.")

## 10. Visually inspect one usable image per folder

This catches obvious label/folder mistakes before training.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 3, figsize=(10, 7))
for axis, code in zip(axes.flat, CLASS_INFO):
    sample_row = next(row for row in eligible_by_code[code] if not row["excluded"])
    with Image.open(DATASET_ROOT / sample_row["path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"{code}: {sample_row['class_name']} / {sample_row['surface']}")
    axis.axis("off")

figure.suptitle("SDNET2018 audit samples")
figure.tight_layout()
plt.show()

## 11. Save the Stage 2 summary

The summary separates observed facts from the unresolved parent-lineage gate.

In [ ]:
summary = {
    "stage": "SDNET2018 audit and manifest",
    "official_record": OFFICIAL_RECORD_URL,
    "archive_md5": archive_md5,
    "archive_sha256": archive_sha256,
    "dataset_root": str(DATASET_ROOT.resolve()),
    "expected_counts": EXPECTED_COUNTS,
    "observed_counts": observed_counts,
    "total_images": len(rows),
    "corrupt_images": corrupt_count,
    "wrong_size_images": wrong_size_count,
    "non_rgb_images": non_rgb_count,
    "duplicate_groups": len(duplicate_groups),
    "duplicate_files_excluded": duplicate_file_count,
    "eligible_images": len(eligible_rows),
    "verified_parent_ids": parent_coverage,
    "research_split_status": (
        "parent_mapping_complete" if parent_coverage == len(eligible_rows)
        else "blocked_until_verified_parent_mapping"
    ),
    "development_images": len(development_rows),
    "development_split_basis": "tile_development_only",
    "audit_manifest": str(AUDIT_MANIFEST_PATH.resolve()),
    "development_manifest": str(DEVELOPMENT_MANIFEST_PATH.resolve()),
}

SUMMARY_PATH = OUTPUT_DIR / "stage_two_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Saved", SUMMARY_PATH.resolve())

## Stage 2 exit criterion

Stage 2 is complete when the checksum passes, all six class counts are reported, the full image audit finishes, the sample panel looks correctly labelled, and audit_manifest.csv, development_manifest.csv, and stage_two_summary.json exist.

The next stage may train MobileViT-XS on development_manifest.csv. Results must retain the tile_development_only warning until a complete verified parent mapping is available.

# Stage 3 — Fine-tune MobileViT-XS

This notebook trains the pretrained Stage 1 model on the deterministic Stage 2 development manifest. It performs a frozen-head warm-up followed by full-model fine-tuning and retains the checkpoint with the best validation F1.

> **Scope:** these are preliminary pipeline-validation results. The manifest uses a tile-level split and may contain parent-photograph leakage; do not report it as a Phase 3 research result.

## 1. Configuration

The dataset paths come from Stage 2. Keep the same Colab runtime, or rerun Stage 2 after a runtime reset.

In [ ]:
from collections import Counter
from contextlib import nullcontext
import csv
import json
from pathlib import Path
import random

import numpy as np
from PIL import Image
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
import timm
from timm.data import create_transform, resolve_model_data_config
from timm.data.transforms import str_to_interp_mode
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
MODEL_NAME = "mobilevit_xs.cvnets_in1k"
BATCH_SIZE = 32
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4

AUDIT_DIR = Path("outputs/sdnet2018_audit")
MANIFEST_PATH = AUDIT_DIR / "development_manifest.csv"
STAGE_TWO_SUMMARY_PATH = AUDIT_DIR / "stage_two_summary.json"
OUTPUT_DIR = Path("outputs/mobilevit_xs/development_training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print({"device": str(DEVICE), "torch": torch.__version__, "timm": timm.__version__})

## 2. Load and verify the Stage 2 development split

In [ ]:
if not MANIFEST_PATH.exists() or not STAGE_TWO_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Stage 2 outputs are missing. Run the Stage 2 cells above first."
    )

stage_two_summary = json.loads(STAGE_TWO_SUMMARY_PATH.read_text(encoding="utf-8"))
dataset_root = Path(stage_two_summary["dataset_root"])
with MANIFEST_PATH.open(newline="", encoding="utf-8") as stream:
    rows = list(csv.DictReader(stream))

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root no longer exists: {dataset_root}. Rerun Stage 2.")
if not rows or {row["split_basis"] for row in rows} != {"tile_development_only"}:
    raise ValueError("Expected the Stage 2 tile_development_only manifest.")

train_rows = [row for row in rows if row["split"] == "dev_train"]
validation_rows = [row for row in rows if row["split"] == "dev_validation"]
assert train_rows and validation_rows
assert not ({row["path"] for row in train_rows} & {row["path"] for row in validation_rows})

counts = Counter((row["split"], int(row["label"]), row["surface"]) for row in rows)
for key, value in sorted(counts.items()):
    print(key, value)
print(f"Train: {len(train_rows):,} | Validation: {len(validation_rows):,}")
print("WARNING: tile_development_only; possible parent-photo leakage.")

## 3. Resolve checkpoint preprocessing and build loaders

Resize, crop, interpolation, normalization, and channel handling come from the checkpoint configuration. Training adds only the planned crack-safe augmentation.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=1)
data_config = resolve_model_data_config(model)

base_train_transform = create_transform(
    **data_config,
    is_training=True,
    scale=(0.85, 1.0),
    hflip=0.5,
    vflip=0.5,
    color_jitter=0.2,
)
train_transform = transforms.Compose([
    transforms.RandomRotation(
        15, interpolation=str_to_interp_mode(data_config["interpolation"])
    ),
    base_train_transform,
])
validation_transform = create_transform(**data_config, is_training=False)

class CrackDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        row = self.items[index]
        with Image.open(dataset_root / row["path"]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.tensor(float(row["label"])), row["path"]

train_loader = DataLoader(
    CrackDataset(train_rows, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    CrackDataset(validation_rows, validation_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

sample_images, sample_labels, _ = next(iter(train_loader))
print(json.dumps(data_config, indent=2, default=list))
print("Batch:", tuple(sample_images.shape), tuple(sample_labels.shape))
assert tuple(sample_images.shape[1:]) == tuple(data_config["input_size"])

## 4. Loss, metrics, and one training loop

`pos_weight` is calculated from the training partition only. No balanced sampler is combined with the weighted loss.

In [ ]:
train_positive = sum(int(row["label"]) == 1 for row in train_rows)
train_negative = len(train_rows) - train_positive
pos_weight = torch.tensor([train_negative / train_positive], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
model = model.to(DEVICE)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

def amp_context():
    return torch.amp.autocast("cuda") if DEVICE.type == "cuda" else nullcontext()

def binary_metrics(labels, probabilities):
    predictions = (probabilities >= 0.5).astype(np.int64)
    return {
        "f1": f1_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "precision": precision_score(labels, predictions, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "pr_auc": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
    }

def run_epoch(loader, optimizer=None, head_only=False):
    training = optimizer is not None
    if training and head_only:
        model.eval()
        model.get_classifier().train()
    elif training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_labels, all_probabilities = [], []
    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), amp_context():
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())

    labels_array = np.asarray(all_labels, dtype=np.int64)
    probabilities_array = np.asarray(all_probabilities)
    return total_loss / len(loader.dataset), binary_metrics(labels_array, probabilities_array)

print({"train_positive": train_positive, "train_negative": train_negative, "pos_weight": pos_weight.item()})

## 5. Phase 1 — train the new classifier head for three epochs

In [ ]:
CHECKPOINT_PATH = OUTPUT_DIR / "best_mobilevit_xs.pt"
HISTORY_PATH = OUTPUT_DIR / "history.json"
history = []
best_f1 = -1.0

def train_phase(name, epochs, learning_rate, head_only):
    global best_f1
    for parameter in model.parameters():
        parameter.requires_grad = not head_only
    if head_only:
        for parameter in model.get_classifier().parameters():
            parameter.requires_grad = True

    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=learning_rate, weight_decay=WEIGHT_DECAY,
    )
    for epoch in range(1, epochs + 1):
        train_loss, train_metrics = run_epoch(train_loader, optimizer, head_only=head_only)
        validation_loss, validation_metrics = run_epoch(validation_loader)
        record = {
            "phase": name, "epoch": epoch,
            "train_loss": train_loss, "validation_loss": validation_loss,
            "train_metrics": train_metrics,
            "validation_metrics": validation_metrics,
        }
        history.append(record)
        print(
            f"{name} {epoch}/{epochs} | train loss {train_loss:.4f} | "
            f"val loss {validation_loss:.4f} | val F1 {validation_metrics['f1']:.4f} | "
            f"val recall {validation_metrics['recall']:.4f}"
        )
        if validation_metrics["f1"] > best_f1:
            best_f1 = validation_metrics["f1"]
            torch.save({
                "model_name": MODEL_NAME,
                "model_state_dict": model.state_dict(),
                "data_config": data_config,
                "validation_metrics": validation_metrics,
                "phase": name, "epoch": epoch, "seed": SEED,
                "split_basis": "tile_development_only",
            }, CHECKPOINT_PATH)
        HISTORY_PATH.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")

train_phase("head", HEAD_EPOCHS, HEAD_LR, head_only=True)
print(f"Best validation F1 after head training: {best_f1:.4f}")

## 6. Phase 2 — unfreeze and fine-tune the complete model

In [ ]:
train_phase("full", FINETUNE_EPOCHS, FINETUNE_LR, head_only=False)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print("Saved best checkpoint:", CHECKPOINT_PATH.resolve())
print(json.dumps(checkpoint["validation_metrics"], indent=2))
print("WARNING: tile_development_only; possible parent-photo leakage; not a Phase 3 result.")

## Stage 3 exit criterion

Stage 3 is complete when all eight epochs finish, `best_mobilevit_xs.pt` and `history.json` exist, validation loss and F1 are printed, and the output remains labelled `tile_development_only`. Stage 4 will load the saved checkpoint and produce the presentation figures and error analysis.

# Stage 4 — MobileViT presentation artifacts

This notebook evaluates the best Stage 3 checkpoint once and creates the four presentation artifacts: model flow, training curves, confusion matrix, and representative predictions.

> **Required label:** preliminary pipeline validation only; tile-level splitting may contain parent-photograph leakage and is not a Phase 3 result.

## 1. Load Stage 2 and Stage 3 artifacts

Keep the Stage 3 Colab runtime alive. If it was reset, rerun Stages 2 and 3 because `/content` storage is temporary.

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np
from PIL import Image
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
import timm
from timm.data import create_transform
import torch
from torch.utils.data import DataLoader, Dataset

AUDIT_DIR = Path("outputs/sdnet2018_audit")
TRAINING_DIR = Path("outputs/mobilevit_xs/development_training")
OUTPUT_DIR = Path("outputs/mobilevit_xs/presentation")
MANIFEST_PATH = AUDIT_DIR / "development_manifest.csv"
STAGE_TWO_SUMMARY_PATH = AUDIT_DIR / "stage_two_summary.json"
CHECKPOINT_PATH = TRAINING_DIR / "best_mobilevit_xs.pt"
HISTORY_PATH = TRAINING_DIR / "history.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [MANIFEST_PATH, STAGE_TWO_SUMMARY_PATH, CHECKPOINT_PATH, HISTORY_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing earlier-stage artifacts: {missing}")

stage_two_summary = json.loads(STAGE_TWO_SUMMARY_PATH.read_text(encoding="utf-8"))
history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
dataset_root = Path(stage_two_summary["dataset_root"])
with MANIFEST_PATH.open(newline="", encoding="utf-8") as stream:
    validation_rows = [
        row for row in csv.DictReader(stream) if row["split"] == "dev_validation"
    ]

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root no longer exists: {dataset_root}")
if not validation_rows:
    raise ValueError("The validation partition is empty.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print({"device": str(DEVICE), "validation_images": len(validation_rows), "output_dir": str(OUTPUT_DIR.resolve())})

## 2. Restore the best checkpoint and evaluate once

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model = timm.create_model(checkpoint["model_name"], pretrained=False, num_classes=1)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
validation_transform = create_transform(**checkpoint["data_config"], is_training=False)

class ValidationDataset(Dataset):
    def __len__(self):
        return len(validation_rows)

    def __getitem__(self, index):
        row = validation_rows[index]
        with Image.open(dataset_root / row["path"]) as image:
            tensor = validation_transform(image.convert("RGB"))
        return tensor, int(row["label"]), row["path"], row["surface"]

loader = DataLoader(
    ValidationDataset(), batch_size=64, shuffle=False, num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

prediction_rows = []
with torch.inference_mode():
    for images, labels, paths, surfaces in loader:
        probabilities = torch.sigmoid(model(images.to(DEVICE)).squeeze(1)).cpu().numpy()
        for path, surface, label, probability in zip(paths, surfaces, labels.numpy(), probabilities):
            prediction_rows.append({
                "path": path, "surface": surface, "label": int(label),
                "probability": float(probability),
                "prediction": int(probability >= 0.5),
            })

labels = np.array([row["label"] for row in prediction_rows])
probabilities = np.array([row["probability"] for row in prediction_rows])
predictions = np.array([row["prediction"] for row in prediction_rows])
matrix = confusion_matrix(labels, predictions)
metrics = {
    "f1": f1_score(labels, predictions),
    "recall": recall_score(labels, predictions),
    "precision": precision_score(labels, predictions),
    "balanced_accuracy": balanced_accuracy_score(labels, predictions),
    "pr_auc": average_precision_score(labels, probabilities),
    "roc_auc": roc_auc_score(labels, probabilities),
    "confusion_matrix": matrix.tolist(),
    "threshold": 0.5,
    "split_basis": "tile_development_only",
}

with (OUTPUT_DIR / "validation_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=prediction_rows[0].keys())
    writer.writeheader()
    writer.writerows(prediction_rows)
(OUTPUT_DIR / "validation_metrics.json").write_text(
    json.dumps(metrics, indent=2) + "\n", encoding="utf-8"
)

comparison_keys = ["f1", "recall", "precision", "balanced_accuracy", "pr_auc", "roc_auc"]
metric_deltas = {
    key: abs(metrics[key] - checkpoint["validation_metrics"][key])
    for key in comparison_keys
}
print(json.dumps(metrics, indent=2))
print("Largest difference from saved Stage 3 metrics:", max(metric_deltas.values()))
if max(metric_deltas.values()) > 1e-4:
    print("WARNING: evaluation differs from the saved checkpoint metrics; inspect the runtime and inputs.")
print("Saved per-image predictions and metrics.")

## 3. Training and validation curves

In [ ]:
epochs = np.arange(1, len(history) + 1)
train_loss = [row["train_loss"] for row in history]
validation_loss = [row["validation_loss"] for row in history]
validation_f1 = [row["validation_metrics"]["f1"] for row in history]
validation_recall = [row["validation_metrics"]["recall"] for row in history]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, train_loss, marker="o", label="Train")
axes[0].plot(epochs, validation_loss, marker="o", label="Validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Weighted BCE")
axes[0].legend()
axes[1].plot(epochs, validation_f1, marker="o", label="F1")
axes[1].plot(epochs, validation_recall, marker="o", label="Crack recall")
axes[1].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1))
axes[1].legend()
for axis in axes:
    axis.axvline(3.5, color="gray", linestyle="--", alpha=0.7)
    axis.grid(alpha=0.25)
figure.suptitle("MobileViT-XS: head warm-up (1–3) → full fine-tuning (4–8)")
figure.tight_layout()
curve_path = OUTPUT_DIR / "training_curves.png"
figure.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", curve_path.resolve())

## 4. Validation confusion matrix

In [ ]:
figure, axis = plt.subplots(figsize=(5.4, 4.8))
image = axis.imshow(matrix, cmap="Blues")
for row in range(2):
    for column in range(2):
        axis.text(column, row, f"{matrix[row, column]:,}", ha="center", va="center", fontsize=16)
axis.set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=["No crack", "Crack"], yticklabels=["No crack", "Crack"],
    xlabel="Predicted label", ylabel="True label",
    title=f"Validation confusion matrix\nCrack recall {metrics['recall']:.3f} | F1 {metrics['f1']:.3f}",
)
figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
confusion_path = OUTPUT_DIR / "confusion_matrix.png"
figure.savefig(confusion_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", confusion_path.resolve())

## 5. Correct predictions, false positives, and false negatives

In [ ]:
correct = sorted(
    (row for row in prediction_rows if row["label"] == row["prediction"]),
    key=lambda row: abs(row["probability"] - 0.5), reverse=True,
)
false_positives = sorted(
    (row for row in prediction_rows if row["label"] == 0 and row["prediction"] == 1),
    key=lambda row: row["probability"], reverse=True,
)
false_negatives = sorted(
    (row for row in prediction_rows if row["label"] == 1 and row["prediction"] == 0),
    key=lambda row: row["probability"],
)
groups = [("Correct", correct), ("False positive", false_positives), ("False negative", false_negatives)]

figure, axes = plt.subplots(3, 3, figsize=(10, 10))
for column, (group_name, group_rows) in enumerate(groups):
    for row_index, axis in enumerate(axes[:, column]):
        if row_index >= len(group_rows):
            axis.axis("off")
            continue
        item = group_rows[row_index]
        with Image.open(dataset_root / item["path"]) as source_image:
            axis.imshow(source_image.convert("RGB"))
        axis.set_title(
            f"{group_name}\n{item['surface']} | p(crack)={item['probability']:.3f}",
            fontsize=10,
        )
        axis.axis("off")
figure.suptitle("MobileViT-XS validation examples")
figure.tight_layout()
examples_path = OUTPUT_DIR / "prediction_examples.png"
figure.savefig(examples_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", examples_path.resolve())

## 6. MobileViT feature-flow diagram

In [ ]:
feature_model = timm.create_model(checkpoint["model_name"], pretrained=False, features_only=True).to(DEVICE).eval()
dummy = torch.zeros(1, *checkpoint["data_config"]["input_size"], device=DEVICE)
with torch.inference_mode():
    feature_shapes = [list(feature.shape) for feature in feature_model(dummy)]
del feature_model, dummy

flow_labels = ["Input\n3×256×256"] + [
    f"Stage {index}\n{shape[1]}×{shape[2]}×{shape[3]}"
    for index, shape in enumerate(feature_shapes, start=1)
] + ["Global pool\n+ 1 logit"]
colors = ["#E8F1FB"] + ["#DDF3E4"] * len(feature_shapes) + ["#FBE8D5"]
figure, axis = plt.subplots(figsize=(14, 3.2))
axis.set_xlim(0, len(flow_labels) * 2.1)
axis.set_ylim(0, 3)
axis.axis("off")
for index, (label, color) in enumerate(zip(flow_labels, colors)):
    x = index * 2.1 + 0.2
    box = FancyBboxPatch(
        (x, 0.9), 1.65, 1.15, boxstyle="round,pad=0.08",
        facecolor=color, edgecolor="#34495E", linewidth=1.3,
    )
    axis.add_patch(box)
    axis.text(x + 0.825, 1.475, label, ha="center", va="center", fontsize=10)
    if index < len(flow_labels) - 1:
        axis.annotate("", xy=(x + 2.02, 1.475), xytext=(x + 1.67, 1.475), arrowprops={"arrowstyle": "->"})
axis.set_title("MobileViT-XS feature flow (observed tensor shapes)", fontsize=14)
figure.tight_layout()
flow_path = OUTPUT_DIR / "mobilevit_feature_flow.png"
figure.savefig(flow_path, dpi=180, bbox_inches="tight")
plt.show()
print("Feature shapes:", feature_shapes)
print("Saved", flow_path.resolve())

## 7. Save the presentation summary

In [ ]:
summary = {
    "model": checkpoint["model_name"],
    "checkpoint_phase": checkpoint["phase"],
    "checkpoint_epoch": checkpoint["epoch"],
    "validation_images": len(validation_rows),
    "metrics": metrics,
    "feature_shapes": feature_shapes,
    "artifacts": {
        "feature_flow": str(flow_path.resolve()),
        "training_curves": str(curve_path.resolve()),
        "confusion_matrix": str(confusion_path.resolve()),
        "prediction_examples": str(examples_path.resolve()),
        "predictions": str((OUTPUT_DIR / "validation_predictions.csv").resolve()),
    },
    "limitation": "Preliminary pipeline validation only; tile-level splitting may contain parent-photograph leakage and is not a Phase 3 result.",
}
summary_path = OUTPUT_DIR / "presentation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

for name, path in summary["artifacts"].items():
    assert Path(path).exists(), f"Missing {name}: {path}"
print(json.dumps(summary, indent=2))
print("Stage 4 complete. Saved", summary_path.resolve())

## Stage 4 exit criterion

Stage 4 is complete when all four PNG figures render, their files and per-image predictions are saved, the evaluation metrics match the best Stage 3 checkpoint, and every presentation use includes the preliminary-results limitation.

# Stage 5 — Understand the trained MobileViT

This notebook inspects the actual trained `timm` model instead of recreating MobileViT from diagrams. It records parameters and observed tensor shapes for every MobileViT block, then produces a block-anatomy diagram for the presentation and the later CrackFusion fork decision.

## 1. Restore the trained model and one real validation tile

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image
import timm
from timm.data import create_transform
import torch

AUDIT_DIR = Path("outputs/sdnet2018_audit")
TRAINING_DIR = Path("outputs/mobilevit_xs/development_training")
OUTPUT_DIR = Path("outputs/mobilevit_xs/architecture_inspection")
CHECKPOINT_PATH = TRAINING_DIR / "best_mobilevit_xs.pt"
SUMMARY_PATH = AUDIT_DIR / "stage_two_summary.json"
MANIFEST_PATH = AUDIT_DIR / "development_manifest.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [str(path) for path in [CHECKPOINT_PATH, SUMMARY_PATH, MANIFEST_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing earlier-stage artifacts: {missing}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model = timm.create_model(checkpoint["model_name"], pretrained=False, num_classes=1)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
transform = create_transform(**checkpoint["data_config"], is_training=False)

stage_two_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
dataset_root = Path(stage_two_summary["dataset_root"])
with MANIFEST_PATH.open(newline="", encoding="utf-8") as stream:
    sample_row = next(row for row in csv.DictReader(stream) if row["split"] == "dev_validation")
with Image.open(dataset_root / sample_row["path"]) as image:
    batch = transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)

print({"device": str(DEVICE), "sample": sample_row["path"], "input_shape": list(batch.shape)})

## 2. Locate MobileViT blocks from the running model

No stage names are hard-coded. Blocks are selected from their runtime class, so this report describes the exact checkpoint implementation.

In [ ]:
mobilevit_blocks = [
    (name, module)
    for name, module in model.named_modules()
    if "mobilevitblock" in module.__class__.__name__.lower()
]
if not mobilevit_blocks:
    raise RuntimeError("No MobileViTBlock modules were found in the loaded model.")

total_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Model: {model.__class__.__name__} | parameters: {total_parameters:,}")
for name, block in mobilevit_blocks:
    parameters = sum(parameter.numel() for parameter in block.parameters())
    children = [f"{child_name}:{child.__class__.__name__}" for child_name, child in block.named_children()]
    print(f"{name} | {block.__class__.__name__} | {parameters:,} parameters")
    print("  " + " -> ".join(children))

## 3. Record real execution order and tensor shapes

Hooks observe the block boundaries and their direct components during one inference pass. Patch unfold/fold operations may be functional operations rather than named modules, so they are represented in the conceptual diagram in Section 5.

In [ ]:
def tensor_shapes(value):
    if isinstance(value, torch.Tensor):
        return [list(value.shape)]
    if isinstance(value, (tuple, list)):
        shapes = []
        for item in value:
            shapes.extend(tensor_shapes(item))
        return shapes
    return []

block_records = {}
component_execution = []
handles = []

def block_hook(name):
    def capture(module, inputs, output):
        block_records[name] = {
            "type": module.__class__.__name__,
            "input_shapes": tensor_shapes(inputs),
            "output_shapes": tensor_shapes(output),
            "parameters": sum(parameter.numel() for parameter in module.parameters()),
        }
    return capture

def component_hook(block_name, component_name):
    def capture(module, inputs, output):
        component_execution.append({
            "block": block_name,
            "component": component_name,
            "type": module.__class__.__name__,
            "input_shapes": tensor_shapes(inputs),
            "output_shapes": tensor_shapes(output),
            "parameters": sum(parameter.numel() for parameter in module.parameters()),
        })
    return capture

for block_name, block in mobilevit_blocks:
    handles.append(block.register_forward_hook(block_hook(block_name)))
    for component_name, component in block.named_children():
        handles.append(component.register_forward_hook(component_hook(block_name, component_name)))

with torch.inference_mode():
    logit = model(batch)
for handle in handles:
    handle.remove()

assert len(block_records) == len(mobilevit_blocks)
print("Output logit:", logit.item())
for name, record in block_records.items():
    print(f"{name}: {record['input_shapes']} -> {record['output_shapes']} | {record['parameters']:,} parameters")
print("Observed direct-component executions:", len(component_execution))

## 4. Parameters per top-level stage and per MobileViT block

In [ ]:
stage_parameter_rows = [
    {
        "stage": name,
        "type": module.__class__.__name__,
        "parameters": sum(parameter.numel() for parameter in module.parameters()),
    }
    for name, module in model.named_children()
]
assert sum(row["parameters"] for row in stage_parameter_rows) == total_parameters

with (OUTPUT_DIR / "stage_parameters.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["stage", "type", "parameters"])
    writer.writeheader()
    writer.writerows(stage_parameter_rows)

with (OUTPUT_DIR / "mobilevit_components.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=["block", "component", "type", "input_shapes", "output_shapes", "parameters"],
    )
    writer.writeheader()
    writer.writerows(component_execution)

print(f"{'Stage':<20}{'Type':<24}{'Parameters':>14}")
for row in stage_parameter_rows:
    print(f"{row['stage']:<20}{row['type']:<24}{row['parameters']:>14,}")

## 5. MobileViT block anatomy

A MobileViT block keeps a local convolutional path, projects the feature map into a token-friendly representation, performs global Transformer processing, folds the tokens back to a spatial map, and fuses them with local features. The labels below use the exact first block shapes observed above.

In [ ]:
first_name = next(iter(block_records))
first_record = block_records[first_name]
input_shape = first_record["input_shapes"][0]
output_shape = first_record["output_shapes"][0]
steps = [
    (f"Input map\n{input_shape[1]}×{input_shape[2]}×{input_shape[3]}", "#E8F1FB"),
    ("Local convolution\nspatial texture", "#DDF3E4"),
    ("Unfold patches\nfeature tokens", "#FFF2CC"),
    ("Transformer\nglobal context", "#EADCF8"),
    ("Fold to map\nspatial alignment", "#FFF2CC"),
    ("Concat + convolution\nlocal/global fusion", "#FADBD8"),
    (f"Output map\n{output_shape[1]}×{output_shape[2]}×{output_shape[3]}", "#E8F1FB"),
]

figure, axis = plt.subplots(figsize=(15, 3.2))
axis.set_xlim(0, len(steps) * 2.1)
axis.set_ylim(0, 3)
axis.axis("off")
for index, (label, color) in enumerate(steps):
    x = index * 2.1 + 0.2
    axis.add_patch(FancyBboxPatch(
        (x, 0.9), 1.65, 1.15, boxstyle="round,pad=0.08",
        facecolor=color, edgecolor="#34495E", linewidth=1.3,
    ))
    axis.text(x + 0.825, 1.475, label, ha="center", va="center", fontsize=9.5)
    if index < len(steps) - 1:
        axis.annotate("", xy=(x + 2.02, 1.475), xytext=(x + 1.67, 1.475), arrowprops={"arrowstyle": "->"})
axis.set_title(f"{first_name}: MobileViT local/global processing and fusion", fontsize=14)
figure.tight_layout()
diagram_path = OUTPUT_DIR / "mobilevit_block_anatomy.png"
figure.savefig(diagram_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", diagram_path.resolve())

## 6. Save the inspection report

In [ ]:
report = {
    "model_name": checkpoint["model_name"],
    "total_parameters": total_parameters,
    "sample_path": sample_row["path"],
    "input_shape": list(batch.shape),
    "output_shape": list(logit.shape),
    "mobilevit_block_count": len(mobilevit_blocks),
    "mobilevit_blocks": block_records,
    "top_level_stage_parameters": stage_parameter_rows,
    "component_execution": component_execution,
    "diagram": str(diagram_path.resolve()),
    "interpretation": (
        "Convolution preserves local crack texture; patch-wise Transformer processing adds global context; "
        "the folded global representation is fused back with spatial features."
    ),
}
report_path = OUTPUT_DIR / "architecture_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

for path in [report_path, diagram_path, OUTPUT_DIR / "stage_parameters.csv", OUTPUT_DIR / "mobilevit_components.csv"]:
    assert path.exists(), path
print(json.dumps({
    "mobilevit_blocks": len(mobilevit_blocks),
    "block_names": list(block_records),
    "total_parameters": total_parameters,
    "report": str(report_path.resolve()),
}, indent=2))

In [ ]:
# Download and test the supplied external image
CUSTOM_IMAGE_URL = "https://drive.google.com/file/d/1EDqnAI4QzQUUIBUfeyJ-A_qNbVABP2Po/view?usp=sharing"
CUSTOM_IMAGE_PATH = Path("/content/custom_test_image")
CUSTOM_OUTPUT_DIR = Path("outputs/mobilevit_xs/custom_image_test")
CUSTOM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

downloaded_path = gdown.download(
    url=CUSTOM_IMAGE_URL, output=str(CUSTOM_IMAGE_PATH), quiet=False, fuzzy=True
)
if downloaded_path is None or not CUSTOM_IMAGE_PATH.exists():
    raise RuntimeError("Image download failed. Set Drive access to 'Anyone with the link'.")

try:
    with Image.open(CUSTOM_IMAGE_PATH) as source:
        source.verify()
    with Image.open(CUSTOM_IMAGE_PATH) as source:
        custom_image = source.convert("RGB")
except Exception as error:
    raise ValueError("The downloaded Drive file is not a readable image.") from error

custom_tensor = transform(custom_image).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    custom_probability = torch.sigmoid(model(custom_tensor).squeeze()).item()

custom_prediction = "crack" if custom_probability >= 0.5 else "no crack"
custom_result = {
    "source_url": CUSTOM_IMAGE_URL,
    "prediction": custom_prediction,
    "crack_probability": round(custom_probability, 6),
}
assert 0.0 <= custom_probability <= 1.0
(CUSTOM_OUTPUT_DIR / "prediction.json").write_text(
    json.dumps(custom_result, indent=2) + "\n", encoding="utf-8"
)

print("INPUT")
display(custom_image)
print("OUTPUT", custom_result)


## Random test — one unseen validation image

Run this cell again whenever you want a different image and prediction.

In [ ]:
import random

random_row = random.choice(validation_rows)
with Image.open(dataset_root / random_row["path"]) as source:
    test_image = source.convert("RGB")

test_tensor = transform(test_image).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    crack_probability = torch.sigmoid(model(test_tensor).squeeze()).item()

actual = "crack" if int(random_row["label"]) else "no crack"
predicted = "crack" if crack_probability >= 0.5 else "no crack"
assert 0.0 <= crack_probability <= 1.0

print("INPUT:", random_row["path"])
display(test_image)
print({
    "OUTPUT": predicted,
    "crack_probability": round(crack_probability, 4),
    "actual_label": actual,
    "correct": predicted == actual,
})

## 7. Bundle every generated artifact for local export

This archive includes everything saved under `/content/outputs`: audit/development manifests, summaries, the best checkpoint, training history, validation predictions and metrics, figures, and architecture reports. The original dataset is excluded because it already exists locally.

In [ ]:
import hashlib
import platform
import sys
from zipfile import ZIP_DEFLATED, ZipFile

outputs_root = Path("/content/outputs")
export_path = Path("/content/mobilevit_stages_1_to_5_export.zip")
if not outputs_root.exists():
    raise FileNotFoundError(outputs_root)

runtime_metadata = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "timm": timm.__version__,
    "device": str(DEVICE),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(outputs_root / "runtime_environment.json").write_text(
    json.dumps(runtime_metadata, indent=2) + "\n", encoding="utf-8"
)

artifact_files = sorted(path for path in outputs_root.rglob("*") if path.is_file())
export_manifest = []
for path in artifact_files:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    export_manifest.append({
        "path": path.relative_to(outputs_root.parent).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": digest.hexdigest(),
    })

manifest_path = outputs_root / "export_manifest.json"
manifest_path.write_text(json.dumps(export_manifest, indent=2) + "\n", encoding="utf-8")
artifact_files = sorted(path for path in outputs_root.rglob("*") if path.is_file())
with ZipFile(export_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
    for path in artifact_files:
        archive.write(path, path.relative_to(outputs_root.parent))

with ZipFile(export_path) as archive:
    bad_member = archive.testzip()
assert bad_member is None, f"Corrupt ZIP member: {bad_member}"
print({
    "archive": str(export_path),
    "files": len(artifact_files),
    "size_mib": round(export_path.stat().st_size / 1024**2, 2),
})
print("In the VS Code Colab sidebar, open Contents, right-click this ZIP, and choose Download.")

## Stage 5 exit criterion

Stage 5 is complete when every runtime MobileViT block has an observed input/output shape, execution-order component report, and parameter count; the top-level parameter total matches the complete model; and the block-anatomy diagram renders. The next implementation step is the research-valid baseline or, for architecture prototyping, the explicit CrackFusion-Tiny branch split.

## Final step — download all outputs

The previous cell creates and verifies the ZIP. This cell starts the browser download.


In [ ]:
from google.colab import files

assert export_path.exists(), f"Missing export archive: {export_path}"
files.download(str(export_path))
